In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
appName("spark joins 2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
## /public/trendytech/orders/orders_1gb.csv
## /public/trendytech/retail_db/customers/part-00000

In [3]:
orders_rdd = spark.sparkContext.textFile("/public/trendytech/orders/orders_1gb.csv")

In [4]:
cust_rdd =  spark.sparkContext.textFile("/public/trendytech/retail_db/customers/part-00000")

In [5]:
orders1 = orders_rdd.map(lambda x: (x.split(",")[2],x.split(",")[3]))

In [32]:
# [('11599', 'CLOSED'),
#  ('256', 'PENDING_PAYMENT'),
#  ('12111', 'COMPLETE'),
#  ('8827', 'CLOSED')]

In [6]:
cust1 = cust_rdd.map(lambda x: (x.split(",")[0],x.split(",")[8]) )

In [31]:
#[('1', '78521'), ('2', '80126'), ('3', '00725'), ('4', '92069')]

##### Broadcast a file

In [25]:
cust_broadcast = spark.sparkContext.broadcast(dict(cust1.collect()))

In [26]:
cust_broadcast.value.get("1")

'78521'

In [28]:
def get_pincode(cust_id): ## FUNCTION TO LOCALLY GET THE PINCODE
    try:
        return cust_broadcast.value.get(cust_id)
    except:
        return "-1"

In [29]:
joined_rdd = orders1.map(lambda x: (get_pincode((x[0])),x[1]))  #### Join using a map function

In [30]:
joined_rdd.saveAsTextFile("data/joined_rdd_broadcast")

In [33]:
!hadoop fs -head data/joined_rdd_broadcast/part-00000

('28601', 'CLOSED')
('60625', 'PENDING_PAYMENT')
('95060', 'COMPLETE')
('78240', 'CLOSED')
('00725', 'COMPLETE')
('11237', 'COMPLETE')
('33161', 'COMPLETE')
('00725', 'PROCESSING')
('44107', 'PENDING_PAYMENT')
('38111', 'PENDING_PAYMENT')
('00725', 'PAYMENT_REVIEW')
('00725', 'CLOSED')
('92705', 'PENDING_PAYMENT')
('00725', 'PROCESSING')
('38127', 'COMPLETE')
('00725', 'PENDING_PAYMENT')
('91352', 'COMPLETE')
('33126', 'CLOSED')
('33012', 'PENDING_PAYMENT')
('42101', 'PROCESSING')
('10460', 'PENDING')
('00725', 'COMPLETE')
('65807', 'PENDING_PAYMENT')
('33549', 'CLOSED')
('32822', 'CLOSED')
('92026', 'COMPLETE')
('00725', 'PENDING_PAYMENT')
('91767', 'COMPLETE')
('48126', 'PROCESSING')
('72032', 'PENDING_PAYMENT')
('00725', 'PAYMENT_REVIEW')
('00725', 'COMPLETE')
('00725', 'PENDING_PAYMENT')
('90044', 'PROCESSING')
('75061', 'COMPLETE')
('00725', 'PENDING')
('38018', 'CLOSED')
('70122', 'PROCESSING')
('91343', 'PENDING')
('00725', 'PENDING_PAYMENT')
('63301', 'PENDING_PAYMENT')
('13760